# 01 — O que os dados dizem que o Sandwell não diz

Primeiro passo da fusão, com calma. A ideia toda cabe numa frase:

> **resíduo = medida real − Sandwell**

Se o resíduo é pequeno e aleatório, o Sandwell já está bom ali. Se é grande ou tem padrão, as medidas trazem informação nova.

Rode uma célula de cada vez (**Shift+Enter**) e olhe o resultado antes de passar para a próxima.

## 1. Preparar o ambiente

O notebook fica em `notebooks/`, mas o código do projeto está em `src/`. Esta célula diz ao Python onde procurar.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
print("Raiz do projeto:", RAIZ)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.processing.fusion import load_sandwell

## 2. O Sandwell

Uma grade de ~760 mil valores de anomalia ar-livre (mGal), vinda de satélite.

In [ ]:
sandwell = load_sandwell()
print("latitudes:", len(sandwell.lats), " longitudes:", len(sandwell.lons))
print("valores de %.0f a %.0f mGal" % (sandwell.values.min(), sandwell.values.max()))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(sandwell.lons, sandwell.lats, sandwell.values,
                   cmap="RdBu_r", vmin=-80, vmax=80, shading="auto")
fig.colorbar(im, label="mGal")
ax.set_title("Sandwell V33.1 — anomalia ar-livre")
plt.show()

**Pergunta para você:** onde fica a linha de costa nesse mapa? Dá para ver só pela cor?

## 3. As medidas reais

Duas fontes, já preparadas nos módulos anteriores:
- **marinho**: navios (política `ok`)
- **terrestre**: estações da CPRM e da RGFB

A função abaixo junta tudo e já calcula o resíduo em cada ponto.

In [ ]:
from src.processing.fusion import build_observations

obs, info = build_observations("ok", "add", sandwell)
obs[["source", "lat", "lon", "fa", "sandwell", "r"]].head()

Colunas: `fa` = medida real, `sandwell` = valor do Sandwell no mesmo ponto, `r` = diferença entre os dois.

In [ ]:
obs["source"].value_counts()

## 4. O tamanho do resíduo, por fonte

In [ ]:
obs.groupby("source")["r"].describe().round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for fonte in ["marine", "CPRM", "RGFB"]:
    r = obs.loc[obs["source"] == fonte, "r"]
    ax.hist(r, bins=np.arange(-60, 61, 2), histtype="step", density=True, label=fonte, lw=1.5)
ax.set_xlabel("resíduo = medida − Sandwell (mGal)")
ax.legend()
plt.show()

**Para observar:**
- O histograma do marinho é estreito: no mar, navio e satélite concordam em poucos mGal.
- Os terrestres são bem mais largos: em terra, o Sandwell erra bastante.

Guarde essa diferença: ela é o motivo de o `fusion.py` tratar terra e mar separadamente.

## 5. Onde estão os resíduos grandes?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(obs["lon"], obs["lat"], c=obs["r"], s=1, cmap="PuOr_r", vmin=-30, vmax=30)
fig.colorbar(sc, label="resíduo (mGal)")
ax.set_title("Resíduo em cada medida")
plt.show()

## Para a próxima conversa

Anote o que você viu (ou o que estranhou). O próximo notebook responde: **o resíduo de um ponto ajuda a prever o do vizinho?** Esse é o semivariograma, a peça central da fusão.